# TCC · EDA de Marketing Mix Modeling com Google Meridian

**Dados simulados oficiais, foco GEO × TEMPO.** Não há estimação posterior, ROI ou recomendação de orçamento.

Execute de cima para baixo em runtime novo Python 3.12+ (CPU). A primeira execução instala dependências; isso pode levar alguns minutos. Se o Colab já tinha versões diferentes de NumPy/TensorFlow carregadas, reinicie a sessão após instalar e execute novamente.

O bootstrap clona o repositório público quando necessário, preserva instalações compatíveis e salva tudo na pasta do projeto. A fonte de dados e a API estão fixadas por commit/checksum. Para uma cópia histórica exata do projeto, configure `REPO_REF` com o commit da entrega. Os dados não possuem nomes de canais reais ou localização geográfica real.


In [1]:
# Execute esta célula primeiro. Em Colab novo, usa somente CPU.
import os, sys, subprocess, importlib.util, importlib.metadata
from pathlib import Path

REPO_URL = "https://github.com/thiagossilva17/MMM_Google_Meridian_Project.git"
REPO_REF = "main"  # Para repetir uma execução publicada, substitua pelo SHA do commit.
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
cwd = Path.cwd()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src/config.py").exists()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = cwd / "MMM_Google_Meridian_Project"
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", REPO_REF], check=True)
    elif not (PROJECT_ROOT / "src/config.py").exists():
        raise RuntimeError("Diretório de destino existe, mas não contém o projeto esperado.")

required = ["meridian", "pandas", "numpy", "scipy", "matplotlib", "statsmodels", "tabulate"]
missing = any(importlib.util.find_spec(package) is None for package in required)
try:
    wrong_meridian = importlib.metadata.version("google-meridian") != "2.1.0"
except importlib.metadata.PackageNotFoundError:
    wrong_meridian = True
if missing or wrong_meridian:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_ROOT / "requirements-colab.txt")], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "outputs/.mplcache"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
from src.config import SEED, ROOT
from src.data import load_panel
from src.reporting import show
import numpy as np
np.random.seed(SEED)
geo_time_panel = load_panel()
print(f"Colab: {IN_COLAB}; projeto: {ROOT}; painel: {geo_time_panel.shape}")


Colab: False; projeto: /workspace/scratch/5d823022ed95/tcc-meridian; painel: (6240, 19)


## 01 — Qual é a escala do KPI, do gasto e da atividade de mídia?

### Pergunta
Qual é a escala do KPI, do gasto e da atividade de mídia?

### Motivação
Participações, assimetria, zeros e extremos ajudam a entender o desenho dos dados antes de comparar mercados.

### Método
Somar contagens e gastos por semana; calcular percentis, IQR, CV, fichas por canal, participação no gasto, sequências de atividade e de zeros. Sinalizar IQR 1,5× e |z modificado MAD| > 3,5, preservando observações.

### Interpretação antes de olhar os resultados
Share de gasto mede alocação. Um canal com poucos zeros não é necessariamente eficiente. IQR/MAD são triagem: escala populacional, intermitência e unidades podem explicar extremos.

**Dependências:** painel validado e funções em `src/eda.py`. O código reutilizável está comentado e pode ser aberto para inspecionar as fórmulas. Todos os resultados são calculados na execução, não preenchidos manualmente.


In [2]:
from src.eda import general
report_01 = general(geo_time_panel)
show(report_01)

### Resultado observado e conclusão parcial

O KPI nacional tem média semanal de 422,632,783 conversões sintéticas e CV de 6.9%. O agregado nacional soma conversões entre geos; não soma taxas.

O maior share de gasto é de Channel3 (40.0%). Share mede alocação, não retorno. A atividade varia de 35.7% a 97.3% das células GEO × semana. Exportamos todas as sequências de atividade e zeros, sem classificar com limiares arbitrários.

Foram sinalizados 6039 pares observação/variável por IQR (1,5×) ou MAD (|z modificado|>3,5). São regras de triagem, não critérios de remoção. Séries esparsas e geos grandes podem ser sinalizados legitimamente. Quando MAD é zero o escore fica indefinido. Nenhum extremo foi removido.

### O que ainda NÃO podemos concluir

As relações são descritivas. Não estimamos ROI, contribuição incremental ou efeitos causais.
Os dados são simulados: padrões não descrevem um mercado real e não validam, por si, a identificação de um MMM.

### CHECKPOINT

- **O que descobrimos?** As evidências numéricas acima e as tabelas exportadas respondem à pergunta deste capítulo.
- **Quais problemas apareceram?** Os limites e alertas estão descritos nos resultados; nenhuma observação foi excluída ou imputada.
- **O que falta investigar?** Dinâmica temporal dos sinais e seus controles.
- **Podemos avançar?** Podemos prosseguir com a investigação exploratória após revisar estes achados; isso não aprova automaticamente a modelagem.


**national_descriptive** — 15 linhas; CSV completo em `outputs/tables/`.

,n,mean,median,std,variance,min,max,p01,p05,p25,p50,p75,p95,p99,iqr,cv,zeros,zero_share
conversions,156.0,4.226328e+08,4.227034e+08,2.911830e+07,8.478753e+14,3.402833e+08,4.912874e+08,3.553039e+08,3.736850e+08,4.053507e+08,4.227034e+08,4.404787e+08,4.687004e+08,4.888518e+08,3.512797e+07,0.068897,0.0,0.000000
Channel0_impression,156.0,3.540654e+07,3.496160e+07,1.210007e+07,1.464118e+14,7.851189e+06,6.386554e+07,1.015168e+07,1.569336e+07,2.704481e+07,3.496160e+07,4.495542e+07,5.706419e+07,6.225383e+07,1.791061e+07,0.341747,0.0,0.000000
Channel1_impression,156.0,2.082422e+07,1.937102e+07,1.017229e+07,1.034756e+14,2.729719e+06,5.236681e+07,4.864523e+06,7.073937e+06,1.340884e+07,1.937102e+07,2.684337e+07,4.052737e+07,5.030103e+07,1.343453e+07,0.488484,0.0,0.000000
Channel2_impression,156.0,1.042020e+07,8.122188e+06,9.076909e+06,8.239028e+13,0.000000e+00,4.608557e+07,0.000000e+00,6.173670e+05,3.297471e+06,8.122188e+06,1.566935e+07,2.866586e+07,3.890450e+07,1.237188e+07,0.871088,4.0,0.025641
Channel3_impression,156.0,7.227242e+07,7.085189e+07,2.220926e+07,4.932513e+14,1.796734e+07,1.402645e+08,2.699388e+07,4.020701e+07,5.635578e+07,7.085189e+07,8.414181e+07,1.085364e+08,1.272496e+08,2.778603e+07,0.307299,0.0,0.000000
Channel4_impression,156.0,3.926712e+07,3.773983e+07,1.596923e+07,2.550163e+14,6.971125e+06,8.370790e+07,8.949558e+06,1.448040e+07,2.745158e+07,3.773983e+07,4.983927e+07,6.684530e+07,7.740700e+07,2.238769e+07,0.406682,0.0,0.000000
Channel0_spend,156.0,2.596264e+05,2.563638e+05,8.872649e+04,7.872390e+09,5.757060e+04,4.683083e+05,7.443948e+04,1.150750e+05,1.983121e+05,2.563638e+05,3.296456e+05,4.184359e+05,4.564901e+05,1.313335e+05,0.341747,0.0,0.000000
Channel1_spend,156.0,2.007711e+05,1.867605e+05,9.807345e+04,9.618401e+09,2.631785e+04,5.048806e+05,4.690000e+04,6.820147e+04,1.292777e+05,1.867605e+05,2.588031e+05,3.907338e+05,4.849639e+05,1.295254e+05,0.488484,0.0,0.000000
Channel2_spend,156.0,7.743173e+04,6.035536e+04,6.744983e+04,4.549479e+09,0.000000e+00,3.424584e+05,0.000000e+00,4.587608e+03,2.450326e+04,6.035536e+04,1.164378e+05,2.130138e+05,2.890964e+05,9.193452e+04,0.871088,4.0,0.025641
Channel3_spend,156.0,5.632074e+05,5.521375e+05,1.730732e+05,2.995434e+10,1.400167e+05,1.093059e+06,2.103590e+05,3.133268e+05,4.391716e+05,5.521375e+05,6.557037e+05,8.458071e+05,9.916360e+05,2.165321e+05,0.307299,0.0,0.000000


**channel_summary** — 5 linhas; CSV completo em `outputs/tables/`.

,total_media_units,total_spend,spend_share,active_periods,active_geos,cpmu,n,mean,median,std,...,p05,p25,p50,p75,p95,p99,iqr,cv,zeros,zero_share
channel,,,,,,,,,,,,,,,,,,,,,
Channel0,5523420467,4.050171e+07,0.184524,156,40,0.007333,6240,8.851635e+05,679670.5,8.192795e+05,...,0.0,253645.75,679670.5,1324174.25,2527659.25,3458103.74,1070528.50,0.925569,755,0.120994
Channel1,3248578573,3.132030e+07,0.142694,156,40,0.009641,6240,5.206055e+05,310652.5,6.205734e+05,...,0.0,0.00,310652.5,807552.50,1825045.65,2604638.15,807552.50,1.192022,1742,0.279167
Channel2,1625551369,1.207935e+07,0.055033,152,40,0.007431,6240,2.605050e+05,0.0,5.551397e+05,...,0.0,0.00,0.0,254734.00,1425716.35,2614612.82,254734.00,2.131013,4010,0.642628
Channel3,11274497204,8.786036e+07,0.400289,156,40,0.007793,6240,1.806810e+06,1505379.0,1.304714e+06,...,168567.8,808356.50,1505379.0,2566118.25,4282815.55,5741464.61,1757761.75,0.722109,169,0.027083
Channel4,6125670562,4.773068e+07,0.217459,156,40,0.007792,6240,9.816780e+05,746247.5,9.141036e+05,...,0.0,267594.50,746247.5,1476707.75,2788040.00,3776701.79,1209113.25,0.931164,820,0.131410


...

,active_cells,zero_cells,active_share,active_periods,active_geos
channel,,,,,
Channel0,5485,755,0.879006,156,40
Channel1,4498,1742,0.720833,156,40
Channel2,2230,4010,0.357372,152,40
Channel3,6071,169,0.972917,156,40
Channel4,5420,820,0.868590,156,40


[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]

[Figura gerada; veja outputs/figures ou execute esta célula para exibir.]